# GCSE baseline with GigaChat

This notebook shows the simplest possible baseline for the hackathon:

- read `data/gcse_test.jsonl` and `data/gcse_validate.jsonl`;
- send one question to GigaChat and get one answer back;
- ask the model to score the answer as `exact / partial / incorrect`;
- collect a final table;
- no RAG, no agents, no complicated pipeline.

The idea is simple: first see how far a good prompt and a fair evaluation can take us.


## 1. Preparation

First we make sure the code runs and import the libraries.


In [1]:
import json
from pathlib import Path

import pandas as pd
import requests
import uuid

import warnings
from urllib3.exceptions import InsecureRequestWarning
warnings.filterwarnings("ignore", category=InsecureRequestWarning)

import os

print("Libraries loaded")


Libraries loaded


## 2. Set up GigaChat

Do not store the key inside the notebook. Enter it manually with `getpass`.

If you already have `ACCESS_TOKEN` in environment variables, you can use it, but for the lesson it is clearer to show the full path: `auth key -> access token -> request`.


In [4]:
AUTH_URL = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
BASE_URL = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"
MODEL_NAME = "GigaChat-2-Max"

AUTH_KEY = os.getenv("GIGACHAT_CREDENTIALS")

print("MODEL_NAME:", MODEL_NAME)


MODEL_NAME: GigaChat-2-Max


In [5]:
def get_access_token(auth_key: str) -> str | None:
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json",
        "RqUID": str(uuid.uuid4()),
        "Authorization": f"Basic {auth_key}",
    }
    data = {"scope": "GIGACHAT_API_PERS"}
    response = requests.post(AUTH_URL, headers=headers, data=data, verify=False)
    print("Auth status:", response.status_code)
    if response.status_code != 200:
        print(response.text)
        return None
    return response.json()["access_token"]


ACCESS_TOKEN = get_access_token(AUTH_KEY)
print("ACCESS_TOKEN received:", bool(ACCESS_TOKEN))


Auth status: 200
ACCESS_TOKEN received: True


In [6]:
def ask_model(prompt: str, temperature: float = 0.2, max_tokens: int = 700) -> str | None:
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)
    if response.status_code != 200:
        print("Request error:", response.status_code)
        print(response.text)
        return None
    return response.json()["choices"][0]["message"]["content"]


## 3. Read the data

We only need two fields from the dataset:

- `input` - the question;
- `expected_output` - the reference answer.

We will keep metadata for later analysis.


In [9]:
ROOT = Path("..")

DATA_DIR = ROOT / "data"
TEST_PATH = DATA_DIR / "gcse_test.jsonl"
VALIDATE_PATH = DATA_DIR / "gcse_validate.jsonl"


def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    df["source_file"] = path.name
    df["row_id"] = range(1, len(df) + 1)
    return df


test_df = load_jsonl(TEST_PATH)
validate_df = load_jsonl(VALIDATE_PATH)

print("test:", test_df.shape)
print("validate:", validate_df.shape)
test_df.head(3)


test: (70, 5)
validate: (30, 5)


,input,expected_output,metadata,source_file,row_id
0,A school wants to add several new outdoor devi...,Wireless avoids cabling cost and lets devices ...,"{'id': 'gcse-test-001', 'book': 'gcse', 'chapt...",gcse_test.jsonl,1
1,"Under the SAE taxonomy of vehicle autonomy, wh...",A level 4 ADS is restricted by design to a spe...,"{'id': 'gcse-test-002', 'book': 'gcse', 'chapt...",gcse_test.jsonl,2
2,"Describe how the quicksort algorithm works, in...",This topic is not covered in the textbook.,"{'id': 'gcse-test-003', 'book': 'gcse', 'chapt...",gcse_test.jsonl,3


In [10]:
test_df['metadata'].apply(lambda x: x['chapter']).value_counts()

metadata
2 Fundamentals of programming                                       21
out of scope                                                        10
4 Computer systems                                                   7
3 Fundamentals of data representation                                7
5 Fundamentals of computer networks                                  6
1 Fundamentals of algorithms                                         6
6 Fundamentals of cyber security                                     6
8 Ethical, legal and environmental impacts of digital technology     4
7 Relational databases and structured query language (SQL)           3
Name: count, dtype: int64

## 4. One question, one answer

First we inspect one small example by hand. This helps us understand the model's output format.


In [11]:
SINGLE_PROMPT = """You answer school-level GCSE Computer Science questions.

The subject area is GCSE Computer Science AQA.

Your goal is to give a correct answer that covers the likely textbook and mark scheme points.

Instructions:
- Start with the direct answer.
- For factual or explanation questions, give a broad answer with relevant definitions, examples, details, and related GCSE-level points.
- For algorithm-tracing, calculation, binary, logic gate, table, or step-by-step questions, carefully work through the exact steps and give only the requested result plus a brief explanation.
- Do not add unrelated background if the question asks for a specific final value, order, output, or state.
- Use clear GCSE-level language.

Question:
{question}

Answer:
"""

sample_question = validate_df.loc[0, "input"]
sample_answer = ask_model(SINGLE_PROMPT.format(question=sample_question))
print("Q:", sample_question)
print("\nA:", sample_answer)


Q: The pseudo-code FlagIsTrue = 5 > 6 followed by OUTPUT FlagIsTrue is executed. What is output, and what data type is the result?

A: **Output:** False  
**Data Type:** Boolean

Explanation: The expression `5 > 6` evaluates to false because 5 is less than 6. Therefore, the variable `FlagIsTrue` will be assigned the boolean value `False`, which is then outputted.


## 5. Compare the answer with the reference

Now we ask the model to act as a strict grader.

We want a simple and stable format:

- `exact` - the meaning matches fully;
- `partial` - the answer is close, but incomplete or slightly wrong;
- `incorrect` - the answer does not fit.


In [12]:
from typing import Literal
from pydantic import BaseModel, Field


class JudgeResult(BaseModel):
    label: Literal["exact", "partial", "incorrect"] = Field(
        description="One of: exact, partial, incorrect"
    )
    comment: str = Field(description="Short reason for the decision")


def create_prompt(question: str, reference: str, prediction: str) -> str:
    return f'''
You are a strict grader for a school question.

Compare the student's answer with the reference.

Return only valid JSON.
Do not use markdown.
Do not wrap the answer in ```json.
Use double quotes for all JSON keys and string values.

Required format:
{{
  "label": "exact | partial | incorrect",
  "comment": "short reason"
}}

Rules:
- exact: the answer matches the reference in meaning;
- partial: the answer is partially correct, but incomplete or slightly wrong;
- incorrect: the answer does not match the reference.

Question:
{question}

Reference:
{reference}

Student answer:
{prediction}
'''.strip()


def judge_answer(question: str, reference: str, prediction: str) -> JudgeResult:
    raw = ask_model(
        create_prompt(
            question=question,
            reference=reference,
            prediction=prediction,
        ),
        temperature=0.0,
        max_tokens=200,
    )

    data = JudgeResult.model_validate_json(raw)
    return data

In [13]:
example_judge = judge_answer(
    sample_question,
    validate_df.loc[0, "expected_output"],
    sample_answer,
)

example_judge

JudgeResult(label='exact', comment='Answer fully matches the reference.')

## 6. Run the full set

To keep the notebook simple, we start with a small slice. After that you can remove the limit and run the full file.

If the API is slow, reduce `N`.


In [14]:
import pandas as pd


def run_judge_batch(
    test_df: pd.DataFrame,
    start_idx: int,
    end_idx: int,
) -> pd.DataFrame:
    """
    Runs answer generation and judging for rows test_df.iloc[start_idx:end_idx].

    start_idx is inclusive.
    end_idx is exclusive.
    """

    subset = test_df.iloc[start_idx:end_idx].copy()

    predictions = []
    judgements: list[JudgeResult] = []

    total = len(subset)

    for batch_pos, (row_idx, row) in enumerate(subset.iterrows(), start=1):
        print(f"{batch_pos}/{total} | original index: {row_idx}")

        prompt = SINGLE_PROMPT.format(question=row["input"])

        prediction = ask_model(prompt) or ""

        verdict = judge_answer(
            question=row["input"],
            reference=row["expected_output"],
            prediction=prediction,
        )

        predictions.append(prediction)
        judgements.append(verdict)

    subset["prediction"] = predictions
    subset["judge_label"] = [item.label for item in judgements]
    subset["judge_comment"] = [item.comment for item in judgements]

    return subset

In [15]:
batch0_25 = run_judge_batch(validate_df, 0, 30)

1/30 | original index: 0
2/30 | original index: 1
3/30 | original index: 2
4/30 | original index: 3
5/30 | original index: 4
6/30 | original index: 5
7/30 | original index: 6
8/30 | original index: 7
9/30 | original index: 8
10/30 | original index: 9
11/30 | original index: 10
12/30 | original index: 11
13/30 | original index: 12
14/30 | original index: 13
15/30 | original index: 14
16/30 | original index: 15
17/30 | original index: 16
18/30 | original index: 17
19/30 | original index: 18
20/30 | original index: 19
21/30 | original index: 20
22/30 | original index: 21
23/30 | original index: 22
24/30 | original index: 23
25/30 | original index: 24
26/30 | original index: 25
27/30 | original index: 26
28/30 | original index: 27
29/30 | original index: 28
30/30 | original index: 29


In [70]:
batch25_50 = run_judge_batch(test_df, 25, 50)

1/25 | original index: 25
2/25 | original index: 26
3/25 | original index: 27
4/25 | original index: 28
5/25 | original index: 29
6/25 | original index: 30
7/25 | original index: 31
8/25 | original index: 32
9/25 | original index: 33
10/25 | original index: 34
11/25 | original index: 35
12/25 | original index: 36
13/25 | original index: 37
14/25 | original index: 38
15/25 | original index: 39
16/25 | original index: 40
17/25 | original index: 41
18/25 | original index: 42
19/25 | original index: 43
20/25 | original index: 44
21/25 | original index: 45
22/25 | original index: 46
23/25 | original index: 47
24/25 | original index: 48
25/25 | original index: 49


In [72]:
batch50_70 = run_judge_batch(test_df, 50, 70)

1/20 | original index: 50
2/20 | original index: 51
3/20 | original index: 52
4/20 | original index: 53
5/20 | original index: 54
6/20 | original index: 55
7/20 | original index: 56
8/20 | original index: 57
9/20 | original index: 58
10/20 | original index: 59
11/20 | original index: 60
12/20 | original index: 61
13/20 | original index: 62
14/20 | original index: 63
15/20 | original index: 64
16/20 | original index: 65
17/20 | original index: 66
18/20 | original index: 67
19/20 | original index: 68
20/20 | original index: 69


## 7. Final table

Now we build a small summary.


In [16]:
result_df = pd.concat([batch0_25], axis=0)
result_df["judge_label"].value_counts()

judge_label
exact        20
incorrect     6
partial       4
Name: count, dtype: int64

In [74]:
exact_rate = (result_df["judge_label"] == "exact").mean()
exact_rate

np.float64(0.8857142857142857)

In [17]:
# 1. Сохраняем весь result_df
result_df.to_json(
    DATA_DIR / "gcse_gigachat_baseline_results.json",
    orient="records",
    force_ascii=False,
    indent=2,
)

# 2. Сохраняем только partial + incorrect
failed_df = result_df[result_df["judge_label"].isin(["partial", "incorrect"])].copy()

failed_df.to_json(
    DATA_DIR / "gcse_gigachat_baseline_failed.json",
    orient="records",
    force_ascii=False,
    indent=2,
)

## 8. Error analysis

The baseline model performs well on most general GCSE Computer Science questions. It usually gives broad and relevant explanations for conceptual topics, definitions, advantages/disadvantages, and standard textbook-style questions.

Most of the remaining errors are concentrated in questions that require exact step-by-step execution or exact numerical conventions. In particular, the model is less reliable on algorithm tracing tasks, simple but precise calculations, unit conversions, and cases where the expected answer depends on a specific textbook convention rather than general computer science knowledge.

Another source of errors is over-answering. Because the prompt encourages broad answers, the model sometimes adds correct general information but misses the exact wording or narrow focus expected by the reference answer. This is especially visible when the mark scheme expects a specific category, phrase, or simplified school-level explanation.

Overall, the baseline is strong for broad factual coverage, but weaker when the task requires exact simulation, exact arithmetic, or strict alignment with the textbook’s preferred terminology.